# RAGAs 评估当前项目的 RAG

本 Notebook 使用项目中的 `knowledge_db/prompt_engineering`、M3E、Chroma 和 Gemini，评估两个指标：

- **Faithfulness**：答案中的陈述是否能被检索上下文支持；
- **Answer Relevancy**：答案是否真正回答了用户问题。

In [ ]:
# 为了和本 Notebook 的旧版指标名称保持一致，建议使用：
# %pip install "ragas==0.2.15"

import os
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, UnstructuredMarkdownLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()
knowledge_path = "../knowledge_db/prompt_engineering"
embedding_model = HuggingFaceEmbeddings(model_name="moka-ai/m3e-base")
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=os.environ["GEMINI_API_KEY"],
    temperature=0,
)

In [ ]:
documents = DirectoryLoader(
    knowledge_path,
    glob="**/*.md",
    loader_cls=UnstructuredMarkdownLoader,
).load()
chunks = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=100
).split_documents(documents)
vectorstore = Chroma.from_documents(chunks, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

prompt = ChatPromptTemplate.from_template(
    "只根据上下文回答问题。如果上下文没有答案，请明确说不知道。\n"
    "上下文：{context}\n问题：{question}"
)
answer_chain = prompt | llm | StrOutputParser()

In [ ]:
# 每条样本都必须保存问题、检索上下文和最终答案。
questions = [
    "文本转换主要解决什么问题？",
    "文本转换有哪些常见方法？",
    "请总结文本转换文章的主要观点和示例。",
]

samples = []
for question in questions:
    docs = retriever.invoke(question)
    contexts = [doc.page_content for doc in docs]
    answer = answer_chain.invoke({
        "context": "\n\n".join(contexts),
        "question": question,
    })
    samples.append({
        "user_input": question,
        "retrieved_contexts": contexts,
        "response": answer,
    })

print(samples[0])

In [ ]:
from ragas import EvaluationDataset, evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import AnswerRelevancy, Faithfulness

evaluation_dataset = EvaluationDataset.from_list(samples)
evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embedding_model)

results = evaluate(
    dataset=evaluation_dataset,
    metrics=[
        Faithfulness(llm=evaluator_llm),
        AnswerRelevancy(
            llm=evaluator_llm,
            embeddings=evaluator_embeddings,
        ),
    ],
)
results

## 如何理解分数

- `faithfulness` 低：答案中可能出现了上下文没有支持的内容，优先检查检索结果和 Prompt。
- `answer_relevancy` 低：答案可能跑题、过于笼统，或没有直接回答问题。
- 评估结果依赖问题集质量；正式评估应准备更多覆盖不同难度和主题的问题。

```python
results = evaluate(
    dataset=EvaluationDataset.from_list(samples),
    metrics=[
        Faithfulness(llm=LangchainLLMWrapper(llm)),
        AnswerRelevancy(
            llm=LangchainLLMWrapper(llm),
            embeddings=LangchainEmbeddingsWrapper(embedding_model),
        ),
    ],
)
```

`Faithfulness`

```python
Faithfulness(
    llm=LangchainLLMWrapper(llm),
    max_retries=1,
    name="my_faithfulness",
)
```

| 参数 | 作用 |
|---|---|
| `llm` | 用于判断答案是否被上下文支持 |
| `max_retries` | 评估失败时的最大重试次数 |
| `name` | 自定义指标名称 |

`AnswerRelevancy`

```python
AnswerRelevancy(
    llm=LangchainLLMWrapper(llm),
    embeddings=LangchainEmbeddingsWrapper(embedding_model),
    strictness=3,
    max_retries=2,
    name="my_answer_relevancy",
)
```

| 参数 | 作用 |
|---|---|
| `llm` | 判断答案是否回答了问题 |
| `embeddings` | 计算问题和答案之间的语义相似度 |
| `strictness` | 生成和评估的反向问题数量 |
| `max_retries` | 评估失败时的最大重试次数 |
| `name` | 自定义指标名称 |

```py
results = evaluate(
    dataset=evaluation_dataset,
    metrics=metrics,
    raise_exceptions=False,
    show_progress=True,
)
```

常用的 evaluate() 参数：
- dataset：评估数据集；
- metrics：评估指标；
- llm：统一提供评估模型；
- embeddings：统一提供 Embedding 模型；
- raise_exceptions：是否遇到单条失败就抛出异常；
- show_progress：是否显示评估进度。
